Prétraitement & Feature Engineering
Objectif : transformer les données brutes en features utilisables par le modèle.

In [2]:
# ── Cellule 1 : Imports ────────────────────────────────
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

df_train = pd.read_csv('../data/fraudTrain.csv')
df_test  = pd.read_csv('../data/fraudTest.csv')

In [3]:
# ── Cellule 2 : Suppression des colonnes inutiles ─────
cols_to_drop = ['first', 'last', 'street', 'trans_num', 'zip', 'city']
df_train.drop(columns=cols_to_drop, inplace=True)
df_test.drop(columns=cols_to_drop, inplace=True)

In [4]:
# ── Cellule 3 : Feature Engineering Temporel ──────────
for df in [df_train, df_test]:
    df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
    df['hour']        = df['trans_date_trans_time'].dt.hour
    df['day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
    df['month']       = df['trans_date_trans_time'].dt.month
    df['is_night']    = df['hour'].apply(lambda x: 1 if (x >= 22 or x <= 5) else 0)
    df['dob']         = pd.to_datetime(df['dob'])
    df['age']         = (df['trans_date_trans_time'] - df['dob']).dt.days // 365
    df.drop(columns=['trans_date_trans_time', 'dob', 'unix_time'], inplace=True)

In [5]:
# ── Cellule 4 : Feature Engineering Géographique ──────
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = (sin(dlat/2)**2 +
         cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2)
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

for df in [df_train, df_test]:
    df['distance_km'] = df.apply(
        lambda r: haversine(r['lat'], r['long'], r['merch_lat'], r['merch_long']),
        axis=1
    )

In [6]:
# ── Cellule 5 : Features comportementales par carte ───
# ATTENTION : fit sur train uniquement, transform sur les deux
df_train = df_train.sort_values('cc_num')
df_train['avg_amt_card']   = df_train.groupby('cc_num')['amt'].transform('mean')
df_train['std_amt_card']   = df_train.groupby('cc_num')['amt'].transform('std').fillna(0)
df_train['tx_count_card']  = df_train.groupby('cc_num')['amt'].transform('count')
df_train['amt_deviation']  = df_train['amt'] - df_train['avg_amt_card']

# Pour le test : mapper les stats calculées depuis le train
card_stats = df_train.groupby('cc_num')['amt'].agg(['mean','std','count']).reset_index()
card_stats.columns = ['cc_num','avg_amt_card','std_amt_card','tx_count_card']
df_test = df_test.merge(card_stats, on='cc_num', how='left')
df_test['avg_amt_card'].fillna(df_test['amt'], inplace=True)
df_test['std_amt_card'].fillna(0, inplace=True)
df_test['tx_count_card'].fillna(1, inplace=True)
df_test['amt_deviation'] = df_test['amt'] - df_test['avg_amt_card']

# Supprimer cc_num après feature engineering
df_train.drop(columns=['cc_num'], inplace=True)
df_test.drop(columns=['cc_num'], inplace=True)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_6684\347726838.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_test['avg_amt_card'].fillna(df_test['amt'], inplace=True)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_6684\347726838.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a co

In [7]:
# ── Cellule 6 : Encodage des variables catégorielles ──
# Gender : Label Encoding simple
le_gender = LabelEncoder()
df_train['gender'] = le_gender.fit_transform(df_train['gender'])
df_test['gender']  = le_gender.transform(df_test['gender'])

# Category : One-Hot Encoding
df_train = pd.get_dummies(df_train, columns=['category'], drop_first=False)
df_test  = pd.get_dummies(df_test,  columns=['category'], drop_first=False)
# Aligner les colonnes train/test (important si certaines catégories absentes)
df_test = df_test.reindex(columns=df_train.columns, fill_value=0)

# State, Job, Merchant : Target Encoding
target_cols = ['state', 'job', 'merchant']
for col in target_cols:
    mapping = df_train.groupby(col)['is_fraud'].mean()
    df_train[col] = df_train[col].map(mapping)
    df_test[col]  = df_test[col].map(mapping).fillna(mapping.mean())

In [8]:
# ── Cellule 7 : Séparation X / y ──────────────────────
X_train = df_train.drop(columns=['is_fraud'])
y_train = df_train['is_fraud']
X_test  = df_test.drop(columns=['is_fraud'])
y_test  = df_test['is_fraud']

In [9]:
# ── Cellule 8 : Normalisation ─────────────────────────
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Sauvegarder le scaler pour la plateforme web plus tard
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Sauvegarder les données traitées pour les notebooks suivants
data_to_save = {
    'X_train_scaled': X_train_scaled,
    'X_test_scaled': X_test_scaled,
    'y_train': y_train.values,
    'y_test': y_test.values
}
with open('../data/processed_data.pkl', 'wb') as f:
    pickle.dump(data_to_save, f)

print(f"✅ Train : {X_train_scaled.shape} | Test : {X_test_scaled.shape}")
print(f"   Taux fraude train : {y_train.mean()*100:.2f}%")
print("✅ Données sauvegardées dans '../data/processed_data.pkl'")

✅ Train : (1296675, 35) | Test : (555719, 35)
   Taux fraude train : 0.58%
✅ Données sauvegardées dans '../data/processed_data.pkl'


In [10]:
import json

# Les colonnes exactes attendues (ordre crucial !)
feature_columns = list(X_train.columns)
with open('../models/feature_columns.json', 'w') as f:
    json.dump(feature_columns, f)

print("✅ Colonnes sauvegardées pour la plateforme web")


✅ Colonnes sauvegardées pour la plateforme web
